# 项目 V1.0 正式启动

## **项目名称**：政务系统缺陷严重度智能分类

**项目背景**

你在人社局政务系统驻场运维实习中，每天都会收到窗口人员反馈的各类系统缺陷。这些缺陷有的影响业务办理（严重），有的只是界面显示问题（轻微）。目前缺陷严重度的判断依赖人工经验，效率低且标准不一。

**项目目标**：构建一个机器学习模型，根据缺陷的属性自动判断严重程度，辅助运维团队进行优先级排序和资源调度。

## 第一步：设计数据集字段

根据你的实习经验，一个缺陷记录通常包含以下信息：

| 字段名 | 类型 | 含义 | 示例值 |
|--------|------|------|--------|
| 缺陷类型 | 分类 | 功能缺陷/性能缺陷/界面缺陷 | "功能" |
| 影响模块数 | 数值 | 涉及几个业务模块 | 1, 2, 3 |
| 复现概率 | 分类 | 每次操作必现/偶发 | "必现" |
| 数据影响 | 二值 | 是否影响数据库数据 | 0/1 |
| 业务中断 | 二值 | 是否导致业务无法办理 | 0/1 |
| 严重程度 | 标签 | 致命/严重/一般/轻微 | "严重" |

**为什么选这些字段？** 因为它们都来自你实习中的真实经验，面试时你能讲出每个字段的业务含义。

## 第二步：生成模拟数据（500条）

In [6]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 500

# 缺陷类型
defect_types = np.random.choice(["功能", "性能", "界面"], size=n, p=[0.5, 0.3, 0.2])

# 影响模块数（1-5个）
modules_affected = np.random.randint(1, 6, size=n)

# 复现概率（必现概率高，偶发概率低）
reproduce_prob = np.random.choice(["必现", "偶发"], size=n, p=[0.7, 0.3])

# 数据影响（是否影响数据库）
data_impact = np.random.choice([0, 1], size=n, p=[0.6, 0.4])

# 业务中断（是否导致业务无法办理）
business_interrupt = np.random.choice([0, 1], size=n, p=[0.7, 0.3])

# 根据规则生成严重程度标签
severity = []
for i in range(n):
    score = 0
    # 缺陷类型：功能缺陷通常更严重
    if defect_types[i] == "功能":
        score += 2
    elif defect_types[i] == "性能":
        score += 1
    # 影响模块数越多越严重
    score += modules_affected[i]
    # 必现的缺陷更严重
    if reproduce_prob[i] == "必现":
        score += 2
    else:
        score += 1
    # 影响数据 +2
    if data_impact[i] == 1:
        score += 2
    # 业务中断 +3
    if business_interrupt[i] == 1:
        score += 3

    # 根据总分映射严重程度
    if score >= 10:
        severity.append("致命")
    elif score >= 7:
        severity.append("严重")
    elif score >= 4:
        severity.append("一般")
    else:
        severity.append("轻微")

# 组装成 DataFrame
df = pd.DataFrame({
    "缺陷类型": defect_types,
    "影响模块数": modules_affected,
    "复现概率": reproduce_prob,
    "数据影响": data_impact,
    "业务中断": business_interrupt,
    "严重程度": severity
})

print("======================== 数据预览 ========================")
display(df.head(10))
print(f"\n数据形状：{df.shape}")
print(f"\n严重程度分布：")
print(df["严重程度"].value_counts())

# 保存为 CSV 文件
df.to_csv("defect_data.csv", index=False, encoding="utf-8-sig")
print("数据已保存为 defect_data.csv")

======================== 数据预览 ========================


,缺陷类型,影响模块数,复现概率,数据影响,业务中断,严重程度
0,功能,1,必现,0,0,一般
1,界面,1,偶发,1,1,严重
2,性能,1,必现,0,1,严重
3,性能,5,必现,0,0,严重
4,功能,4,偶发,0,0,严重
5,功能,5,必现,1,0,致命
6,功能,4,偶发,0,0,严重
7,界面,5,必现,0,0,严重
8,性能,5,偶发,0,1,致命
9,性能,3,必现,1,0,严重



数据形状：(500, 6)

严重程度分布：
严重程度
严重    208
一般    155
致命    118
轻微     19
Name: count, dtype: int64
数据已保存为 defect_data.csv
